# Testing Different Extraction Techniques

In [1]:
from pathlib import Path
import sys

# Add repo root so `backend` is importable when running from notebooks/
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [2]:
import pandas as pd

# Load a single row from the merged plain language dataset
dataset_path = repo_root / "data" / "merged_plain_language_dataset.csv"
df = pd.read_csv(dataset_path)
df.iloc[0]

index                                                                  0
original_text          Plants produce a plethora of natural products,...
plain_language_text    Significance Recently discovered biosynthetic ...
split                                                              train
source_dataset                                                     CELLS
Name: 0, dtype: object

In [3]:
# Choosing Medlane to test, because it seems to be most similar to radiology reports, and lets us test on important medical entities (disease names, measurements, etc)

medlane = df[df['source_dataset'] == 'MedLane']
medlane.iloc[0]

index                                                              81680
original_text          known lastname 51946 is a 31-year-old male s/p...
plain_language_text    patient 51946 is a 31-year-old male after a a ...
split                                                              train
source_dataset                                                   MedLane
Name: 81680, dtype: object

In [4]:
print(f"Medical Report: {medlane.iloc[0]['original_text']}")
print(f"Plain Language Text: {medlane.iloc[0]['plain_language_text']}")

Medical Report: known lastname 51946 is a 31-year-old male s/p a a sibling matched allogeneic bone marrow transplant for severe aplastic anemia .
Plain Language Text: patient 51946 is a 31-year-old male after a a sibling matched allogeneic bone marrow transplant for severe aplastic lack of enough healthy red blood cells .


## SpaCy Extractor - en_core_sci_md

In [5]:
import pandas as pd


synthetic_reports = pd.read_csv("synthetic_radiology_reports.csv")
medical_report = synthetic_reports.iloc[0]['full_text']
medical_report

'Examination(s): MR brain without and with contrast. Clinical information: Chronic headaches, recent change in character. Comparison(s): Brain MRI without and with contrast XX/XX/2019. Technique: Multisequence, multiplanar MR of the brain obtained without and with intravenous contrast. Contrast type, dose and administration as documented in electronic medical record.. Findings: Brain parenchyma: No acute diffusion restriction to suggest acute infarct. No abnormal susceptibility to suggest macrohemorrhage. No mass-like enhancing lesion identified. Scattered few tiny foci of T2/FLAIR hyperintensity within the supratentorial deep and periventricular white matter, greatest in the periventricular frontal region, each measuring up to 3 mm, likely chronic microvascular ischemic change given age. No abnormal enhancement.\nMidline/ventricles: Midline structures are midline. Ventricular size and morphology within expected limits for age without transependymal CSF flow.\nExtra-axial/meninges: No 

In [7]:
from backend.src.services.summaries.entity_extraction.spacy import SpacyExtractor

SpacyExtractor.load_model()

extractor = SpacyExtractor()

In [11]:
# New methods
# entities = extractor.extract_entities(medical_report)
# entities
summary = extractor.get_section_summary(medical_report)
summary

{'general': [ClinicalEntity(original_text='MR brain', canonical_name='Brain', definition='The part of CENTRAL NERVOUS SYSTEM that is contained within the skull (CRANIUM). Arising from the NEURAL TUBE, the embryonic brain is comprised of three major parts including PROSENCEPHALON (the forebrain); MESENCEPHALON (the midbrain); and RHOMBENCEPHALON (the hindbrain). The developed brain consists of CEREBRUM; CEREBELLUM; and other structures in the BRAIN STEM.', semantic_types=['T023'], confidence=0.8261552453041077, section=None, is_negated=False, is_uncertain=False, is_family=False, is_historical=False, start_char=16, end_char=24, mesh_id='C0006104', aliases=['Encephalon']),
  ClinicalEntity(original_text='Chronic headaches', canonical_name='Chronic Headache', definition=None, semantic_types=['T047'], confidence=0.9834021329879761, section=None, is_negated=False, is_uncertain=False, is_family=False, is_historical=False, start_char=74, end_char=91, mesh_id='C0151293', aliases=['Headache, Chr

In [ ]:
print("\n=== TEST 1: All entities (including negated) ===")
entities = extractor.extract_entities(medical_report, include_negated=True)
for e in entities:
    print(f"• {e.original_text} -> {e.canonical_name}")
    print(f"  Negated: {e.is_negated}, Uncertain: {e.is_uncertain}, Family: {e.is_family}")
    print(f"  Section: {e.section}")
    print()

print("\n=== TEST 2: Current findings only ===")
findings = extractor.get_findings_only(medical_report)
for e in findings:
    print(f"• {e.canonical_name} (confidence: {e.confidence:.2f})")

print("\n=== TEST 3: Critical findings ===")
critical = extractor.get_critical_findings(medical_report)
for e in critical:
    print(f"• {e.canonical_name}")

print("\n=== TEST 4: Simplification context ===")
print(extractor.build_simplification_context(medical_report))

print("\n=== TEST 5: Glossary ===")
glossary = extractor.get_simplification_glossary(medical_report)
for term, definition in glossary.items():
    print(f"{term}: {definition}")

## MedSpaCy Extractor - Can be used for tagging negation, historical, hypothetical tags to entities

In [ ]:
import spacy
import medspacy
from scispacy.linking import EntityLinker

# 1. Load scispaCy model
nlp = spacy.load("en_core_sci_md")

# 2. Add MedSpaCy components
nlp.add_pipe("medspacy_context")
nlp.add_pipe("medspacy_sectionizer")
nlp.add_pipe("medspacy_postprocessor")

nlp.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "name": "mesh"})

CLINICAL_TYPES = {
    'T047',  # Disease or Syndrome ✓
    'T048',  # Mental or Behavioral Dysfunction ✓
    'T049',  # Cell or Molecular Dysfunction ✓
    'T059',  # Laboratory or Test Result (CHANGED from T122)
    'T060',  # Diagnostic Procedure ✓
    'T061',  # Therapeutic or Preventive Procedure ✓
    'T184',  # Sign or Symptom ✓
    'T033',  # Finding ✓
    'T037',  # Injury or Poisoning ✓
    'T046',  # Pathologic Function ✓
    'T191',  # Neoplastic Process ✓
}

# Test text
test_text = """
FINDINGS:
No evidence of pneumonia.
Possible small nodule in the right lower lobe.
Severe aplastic anemia.

IMPRESSION:
Mother has history of breast cancer.
"""

doc = nlp(test_text)

# Get linker
linker = nlp.get_pipe("scispacy_linker")

print("="*80)
for ent in doc.ents:
    # Check if entity has MeSH links
    if not ent._.kb_ents:
        continue
    
    # Get MeSH entity
    mesh_id, score = ent._.kb_ents[0]
    mesh_entity = linker.kb.cui_to_entity[mesh_id]
    
    # Skip if not in CLINICAL_TYPES
    if not any(t in CLINICAL_TYPES for t in mesh_entity.types):
        continue

    print(f"\nENTITY: '{ent.text}'")
    print(f"    Label: {ent.label_}")
    
    # MedSpaCy attributes
    print(f"MedSpaCy Context:")
    print(f"      Section: {ent._.section_category}")
    print(f"      Negated: {ent._.is_negated}")
    print(f"      Uncertain: {ent._.is_uncertain}")
    print(f"      Family History: {ent._.is_family}")
    print(f"      Historical: {ent._.is_historical}")
    
    # MeSH attributes
    print(f"MeSH Info:")
    print(f"      MeSH ID: {mesh_id}")
    print(f"      Canonical Name: {mesh_entity.canonical_name}")
    print(f"      Confidence: {score:.3f}")
    print(f"      Semantic Types: {mesh_entity.types}")
    print(f"      Definition: {mesh_entity.definition}")
    print(f"      Aliases: {list(mesh_entity.aliases)[:5]}")  # First 5 aliases

In [ ]:
import spacy
import medspacy
from scispacy.linking import EntityLinker
from dataclasses import dataclass
from typing import Optional

CLINICAL_TYPES = {
    'T047',  # Disease or Syndrome
    'T048',  # Mental or Behavioral Dysfunction
    'T049',  # Cell or Molecular Dysfunction
    'T059',  # Laboratory or Test Result
    'T060',  # Diagnostic Procedure
    'T061',  # Therapeutic or Preventive Procedure
    'T184',  # Sign or Symptom
    'T033',  # Finding
    'T037',  # Injury or Poisoning
    'T046',  # Pathologic Function
    'T191',  # Neoplastic Process
}

@dataclass
class ClinicalEntity:
    text: str
    label: str
    section: Optional[str]
    is_negated: bool
    is_uncertain: bool
    is_family: bool
    is_historical: bool
    mesh_id: Optional[str]
    mesh_name: Optional[str]
    mesh_score: Optional[float]
    semantic_types: list[str]
    definition: Optional[str]
    aliases: list[str]


def load_clinical_nlp():
    """Load and configure the clinical NLP pipeline."""
    print("Loading NLP pipeline...")
    nlp = spacy.load("en_core_sci_md")
    print("Spacy model loaded!")
    nlp.add_pipe("medspacy_context")
    nlp.add_pipe("medspacy_sectionizer")
    nlp.add_pipe("medspacy_postprocessor")
    nlp.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "name": "mesh"})
    print("Pipeline loaded!")
    return nlp


def extract_clinical_entities(text: str, nlp) -> list[ClinicalEntity]:
    """Extract clinical entities from text with MeSH linking."""
    doc = nlp(text)
    linker = nlp.get_pipe("scispacy_linker")
    
    entities = []
    
    for ent in doc.ents:
        # Check if entity has MeSH links
        if not ent._.kb_ents:
            continue
        
        # Get MeSH entity
        mesh_id, score = ent._.kb_ents[0]
        mesh_entity = linker.kb.cui_to_entity[mesh_id]
        
        # Skip if not in CLINICAL_TYPES
        if not any(t in CLINICAL_TYPES for t in mesh_entity.types):
            continue
        
        entities.append(ClinicalEntity(
            text=ent.text,
            label=ent.label_,
            section=ent._.section_category,
            is_negated=ent._.is_negated,
            is_uncertain=ent._.is_uncertain,
            is_family=ent._.is_family,
            is_historical=ent._.is_historical,
            mesh_id=mesh_id,
            mesh_name=mesh_entity.canonical_name,
            mesh_score=score,
            semantic_types=list(mesh_entity.types),
            definition=mesh_entity.definition,
            aliases=list(mesh_entity.aliases)[:5]
        ))
    
    return entities


def print_entities(entities: list[ClinicalEntity]):
    """Print extracted entities in a formatted way."""
    print("="*80)
    for ent in entities:
        print(f"\nENTITY: '{ent.text}'")
        print(f"    Label: {ent.label}")
        
        print(f"MedSpaCy Context:")
        print(f"      Section: {ent.section}")
        print(f"      Negated: {ent.is_negated}")
        print(f"      Uncertain: {ent.is_uncertain}")
        print(f"      Family History: {ent.is_family}")
        print(f"      Historical: {ent.is_historical}")
        
        print(f"MeSH Info:")
        print(f"      MeSH ID: {ent.mesh_id}")
        print(f"      Canonical Name: {ent.mesh_name}")
        print(f"      Confidence: {ent.mesh_score:.3f}")
        print(f"      Semantic Types: {ent.semantic_types}")
        print(f"      Definition: {ent.definition}")
        print(f"      Aliases: {ent.aliases}")

In [ ]:
import pandas as pd

def load_reports(csv_path: str) -> pd.DataFrame:
    """Load radiology reports from CSV file."""
    df = pd.read_csv(csv_path)
    
    # Verify full_text column exists
    if 'full_text' not in df.columns:
        raise ValueError(f"CSV must contain 'full_text' column. Found: {df.columns.tolist()}")
    
    print(f"Loaded {len(df)} reports")
    print(f"Columns: {df.columns.tolist()}")
    
    return df


def process_all_reports(csv_path: str, nlp) -> pd.DataFrame:
    """Process all reports and return entity extraction results."""
    # Load reports
    reports_df = load_reports(csv_path)
    
    all_results = []
    
    for idx, row in reports_df.iterrows():
        print(f"Processing report {idx + 1}/{len(reports_df)}...")
        
        # Extract entities
        entities = extract_clinical_entities(row['full_text'], nlp)
        
        # Add to results
        for ent in entities:
            all_results.append({
                "report_id": idx,
                "full_text": row['full_text'],
                "entity_text": ent.text,
                "label": ent.label,
                "section": ent.section,
                "is_negated": ent.is_negated,
                "is_uncertain": ent.is_uncertain,
                "is_family": ent.is_family,
                "is_historical": ent.is_historical,
                "mesh_id": ent.mesh_id,
                "mesh_name": ent.mesh_name,
                "mesh_score": ent.mesh_score,
                "semantic_types": ",".join(ent.semantic_types),
            })
    
    return pd.DataFrame(all_results)

In [ ]:
nlp = load_clinical_nlp()

entities_df = process_all_reports("synthetic_radiology_reports.csv")

entities_df

In [ ]:
results_df = process_reports("synthetic_radiology_reports.csv")

# Save results
results_df.to_csv("extracted_entities.csv", index=False)
print(f"\nExtracted {len(results_df)} entities from reports")
print(f"Saved to extracted_entities.csv")

# ============================================================
# Summary Statistics
# ============================================================

print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

print(f"\nTotal entities: {len(results_df)}")
print(f"Unique MeSH concepts: {results_df['mesh_id'].nunique()}")

print("\n--- Entities by Section ---")
print(results_df["section"].value_counts())

print("\n--- Negated vs Affirmed ---")
print(results_df["is_negated"].value_counts())

print("\n--- Uncertain Findings ---")
print(results_df["is_uncertain"].value_counts())

print("\n--- Top 10 Most Common Entities ---")
print(results_df["mesh_name"].value_counts().head(10))

print("\n--- Entities by Modality ---")
print(results_df.groupby("modality")["entity_text"].count())

# ============================================================
# Show Sample Output
# ============================================================

print("\n" + "="*80)
print("SAMPLE ENTITIES FROM FIRST REPORT")
print("="*80)

first_report_entities = results_df[results_df["report_id"] == 1]
for _, row in first_report_entities.iterrows():
    neg = "[NEG]" if row["is_negated"] else ""
    unc = "[UNC]" if row["is_uncertain"] else ""
    print(f"  {row['entity_text']:30} | {row['mesh_name']:40} {neg} {unc}")

## BiomedNLP Multi NER

In [ ]:
from transformers import pipeline

ner = pipeline(
    "ner",
    model="d4data/biomedical-ner-all",
    aggregation_strategy="simple"
)

text = "Patient has severe aplastic anemia and hypertension"
entities = ner(text)

for e in entities:
    print(f"word: {e['word']}, entity group: {e['entity_group']}, confidence: ({e['score']:.2%})")

In [ ]:
from transformers import pipeline

ner = pipeline(
    "ner",
    model="d4data/biomedical-ner-all",
    aggregation_strategy="average"
)

text = "Patient has severe aplastic anemia and hypertension"
entities = ner(text)

for e in entities:
    print(f"word: {e['word']}, entity group: {e['entity_group']}, confidence: ({e['score']:.2%})")

Decently good when getting one word important medical entities.

## BioBERT Diseases NER

In [ ]:
from transformers import pipeline

ner = pipeline(
    "ner",
    model="alvaroalon2/biobert_diseases_ner",
    aggregation_strategy="average"
)

text = "Patient has severe aplastic anemia and hypertension"
entities = ner(text)

for e in entities:
    print(f"word: {e['word']}, entity group: {e['entity_group']}, confidence: ({e['score']:.2%})")

Very accurate at diseases only.

## ClinicalBERT Extractor

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

ner = pipeline(
    "ner",
    model="samrawal/bert-base-uncased_clinical-ner",
    aggregation_strategy="average"
)

text = "Patient has severe aplastic anemia and hypertension"
entities = ner(text)

for e in entities:
    print(f"{e['word']} → {e['entity_group']} ({e['score']:.2%})")

## MedCAT Extractor (Does NER + Linking + Context -> 2GB and slower)